In [ ]:
from typing import Iterable, Callable
from itertools import chain as iterchain, combinations as itercomb
from functools import reduce
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Loc, Cell, Board
from utils import filt_having, draftborhood, count_finals, count_digits
from topology import Zone, allpeers, visibility
from targeting import Node
from linking import Link, HLink, SLink, Chain
from solving import Resolver, Resolving, Pattern, solver, onceolver
from searching import search_breadth

## Links

Connections between 2 drafts or groups of drafts.

Correspond to logical relation of NAND $A \barwedge B$ or XOR: $A \veebar B$.

Generally, anything logical can be linked, including groups of OR'ed drafts $\bigvee C_i$.


### Strong/Hard links

Criteria:

1. (bi-location) only 2 drafts of same digit in a unit
2. (bi-value) only 2 drafts in a cell
3. (bi-partitions) any 2 (non-overlapping) groups that fully contain all habitants in a unit
    - only makes sense to subdivide units by cross-sectors (triplecells)


In [ ]:
def search_hard_1(board: Board) -> Iterable[HLink]:
    """search for biloc single-value links"""
    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = count_digits(drafts)
        for dig, cnt in counts.items():
            if cnt == 2:
                (c1, c2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(c1, dig), Node.at(c2, dig)))

In [ ]:
def search_hard_2(board: Board):
    """search for intra-cell bivalue links"""
    for cell in filter(lambda c: len(c) == 2, board):
        d1, d2 = cell.digits
        yield HLink((Node.at(cell, d1), Node.at(cell, d2)))

In [ ]:
def search_hard_31(board: Board):
    """search for any bipartitions of each unit (including singular drafts)"""
    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))
        totals = count_digits(drafts)
        census: dict[int, dict[Zone, int]] = {dig: {} for dig, cnt in totals.items() if cnt > 1}

        for sect in Zone.sectors(unit):
            for dig, cnt in count_digits(draftborhood(board, sect)).items():
                if dig in census:
                    census[dig][sect] = cnt

        def targ(zone: Zone, dig: int):
            if census[dig][zone] > 1:
                return Node.at(zone, dig)
            else:
                [cell] = filter(lambda c: c.loc in zone and dig in c, drafts)
                return Node.at(cell, dig)

        for dig, subcounts in census.items():
            sectors = set(subcounts.keys())
            for z1, z2 in itercomb(sectors, 2):
                if (z1 & z2 is None) and subcounts[z1] + subcounts[z2] == totals[dig]:
                    yield HLink((targ(z1, dig), targ(z2, dig)))

### Weak/Soft links

Criteria:

- (bi-val) any 2 drafts in a cell
- (bi-loc) any 2 drafts of same digit in a unit
- (bi-tripl) any 2 triplets in a unit
- (triploc) a triplet and a singular draft of the same digit within shared unit


In [ ]:
def softability_2(n1: Node, n2: Node) -> bool:
    """check if the nodes are bival-soft-linkable"""
    return n1.dig != n2.dig and n1.is_cellular and n2.is_cellular and n1.loc == n2.loc


def softlink_2(n1: Node, n2: Node) -> SLink | None:
    """bi-value soft link if possible"""
    assert n1 != n2

    if softability_2(n1, n2):
        return SLink((n1, n2))

In [ ]:
def softability_31(n1: Node, n2: Node):
    """check if the nodes are biloc-soft-linkable (both cellular and sectoral)"""
    return n1.dig == n2.dig and not (n1.zone & n2.zone) and len(visibility(n1.zone, n2.zone)) > 0


def softlink_31(n1: Node, n2: Node) -> SLink | None:
    """inter-location, both singulars and triplets"""
    assert n1 != n2

    if softability_31(n1, n2):
        return SLink((n1, n2))

In [ ]:
Connecting = Callable[[Node, Node], SLink | None]

## Chains

Alternating inference chains.

$(C_1 \veebar C_2) \cdot (C_2 \barwedge C_3) \cdot (C_3 \veebar C_4) \cdot (C_4 \barwedge C_5) \cdot ...$

Resolvable chains of importance:

- single hardlink $(C_1 \veebar C_2)$
- open hard-ended chain $(C_1 \veebar ... \veebar C_n)$
- closed loop: $(C_1 \veebar ... \veebar C_n) \cdot (C_n \barwedge C_1)$


In [ ]:
from linking import are_alternating


def is_proper_chain(chain: Chain):
    l = len(chain)
    return are_alternating(chain) and (l == 1 or l > 2)

In [ ]:
# assuming chains are alternating and hard-started by construction


def match_loop(chain: Chain):
    if not isinstance(chain[-1], SLink):
        return False
    e1, e2 = chain.edges
    return e1 == e2


def match_rope(chain: Chain):
    if not isinstance(chain[-1], HLink):
        return False
    e1, e2 = chain.edges
    # either inter-cell or non-overlapping
    return (e1.is_cellular and e2.is_cellular and e1.loc == e2.loc) or (e1.zone & e2.zone is None)

### Searching

- searching for all hard inks first
- trying to connect them into chains with possible soft links
- searchng breadth-first to find shortest chains
- trying to make closed loop before going on


In [ ]:
def expand_chain(current: Chain, links: Iterable[HLink], connecting: Connecting) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 1 and len(chain) % 2 == 1:
            closing = connecting(e2, e1)
            if closing:
                yield Chain.extend(chain, closing)

    def extend(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if conn := connecting(e2, x1):
            yield Chain.extend(chain, conn, link)
        if conn := connecting(e2, x2):
            yield Chain.extend(chain, conn, link.reversed())
        if conn := connecting(x2, e1):
            yield Chain.exthead(chain, link, conn)
        if conn := connecting(x1, e1):
            yield Chain.exthead(chain, link.reversed(), conn)

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return all(n.zone & a.zone is None for a in anchors for n in lnk)

    for link in filter(noncycling, links):
        for extended in extend(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [ ]:
def search_chains(current: Board, links: Iterable[HLink], matching: Callable[[Chain], bool], connecting: Connecting, max_length: int = 8):
    counts = count_finals(current)

    def rate(link: Link):
        return min(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain((l,)) for l in links]

    def expanding(chain: Chain):
        yield from expand_chain(chain, links, connecting)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

### Resolving

- unclosed hard-edged => invalidate all visible from both edges
- closed loop => invalidate all visible from edges of each soft link


In [ ]:
def vision_2(board: Board, eye: Node) -> set[Node]:
    """All other digits in the cell are obviously visible"""
    if not eye.is_casual:
        return set()
    viscell = board.get(eye.loc)
    return {Node.at(viscell, d) for d in viscell.digits - eye.digits}

In [ ]:
def vision_13(board: Board, eye: Node) -> set[Node]:
    """All peer drafts visible and linkable from the eye node (either cell or sector)"""
    if not eye.is_singular:
        return set()
    dig = eye.dig
    visborhood = set(draftborhood(board, allpeers(eye.zone)))
    visbornood = {Node.at(c, dig) for c in visborhood if dig in c}
    return set(filter(lambda n: softability_31(n, eye), visbornood))

In [ ]:
def crossvision_123(board: Board, eye1: Node, eye2: Node) -> set[Node]:
    e1vision = vision_13(board, eye1) | vision_2(board, eye1)
    e2vision = vision_13(board, eye2) | vision_2(board, eye2)
    return e1vision & e2vision

In [ ]:
def resolve_rope(board: Board, chain: Chain) -> Pattern | None:
    """Cleanup all spoilers visible from both edges"""
    e1, e2 = chain.edges
    spoilers = crossvision_123(board, e1, e2) - chain.anchors()
    if len(spoilers):
        return Pattern(
            spoilers=spoilers,
            anchors={e1, e2},
            chain=chain,
        )

In [ ]:
def resolve_loop(board: Board, chain: Chain) -> Pattern | None:
    """Cleanup all spoilers visible from both edges of each soft link"""
    links = tuple(filter(lambda lnk: isinstance(lnk, SLink), chain))
    spoilers = reduce(lambda a, b: a | b, (crossvision_123(board, l[0], l[1]) for l in links))
    spoilers -= chain.anchors()
    if len(spoilers):
        return Pattern(
            spoilers=spoilers,
            chain=chain,
        )

#### resolver


In [ ]:
def multichains(current: Board) -> Resolving:
    links = set(search_hard_2(current)) | set(search_hard_31(current))

    searching = search_chains(
        current, links, matching=lambda ch: match_loop(ch) or match_rope(ch), connecting=lambda n1, n2: softlink_31(n1, n2) or softlink_2(n1, n2), max_length=8
    )

    for chain in searching:
        assert is_proper_chain(chain), f"bogus chain: {chain}"

        if match_loop(chain):
            res = resolve_loop(current, chain)
        elif match_rope(chain):
            res = resolve_rope(current, chain)
        else:
            raise AssertionError(f"bogus chain: {chain}")

        if res is not None:
            yield res

# A puzzle


In [ ]:
from solving import solve_silent, solve_logging
from resolvers import naked_singles, hidden_singles, naked_multiples, hidden_multiples, locked_triplets

In [ ]:
from utils import fillempty, picture, parsepic, parsepic_wide

puzzle = parsepic("""
....89...
......17.
6........
.2.3.....
.1......9
.......68
8.9.5....
...7..2..
5........
""")

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
puzzle = await solve_silent(puzzle, naked_singles, hidden_singles)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    naked_singles,
    hidden_singles,
    # onceolver(naked_multiples),
    # onceolver(hidden_multiples),
    # onceolver(locked_triplets),
    onceolver(multichains),
    mute={"naked_singles", "hidden_singles"},
    verbose={"*"},
)

# GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
    :root {
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Bool, Dict
from canvas import SudokuCanvas

In [ ]:
# GUI meta-widget


def click_future(button: w.Button) -> asyncio.Future[bool]:
    # TODO: make it cancellable somehow
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Unicode()
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Node))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for node in self.empties:
                self._highlight_node(node, "pink")

            for lnk in self.links:
                self._highlight_link(lnk, "blue")
            for lnk in self.links:
                for n in lnk:
                    if n.is_cellular:
                        self._highlight_node(n, "blue")
                    else:
                        self._highlight_group(n, "blue")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for cell in self.targets:
                self._highlight_node(cell, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in node.zone:
            for dig in node.digits:
                self._canvas.highlight_segment(loc, dig, color=color)

    def _highlight_cell(self, cell: Cell, color: str):
        for dig in cell.digits:
            self._canvas.highlight_segment(cell.loc, dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        n1, n2 = lnk
        if n1.zone.is_cell and n2.zone.is_cell:
            self._canvas.highlight_link(
                n1.loc,
                n1.dig,
                n2.loc,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            n1locs = tuple(iter(n1.zone))
            l1mid = Loc(
                sum(l.r for l in n1locs) // len(n1locs),
                sum(l.c for l in n1locs) // len(n1locs),
            )
            n2locs = tuple(iter(n2.zone))
            l2mid = Loc(
                sum(l.r for l in n2locs) // len(n2locs),
                sum(l.c for l in n2locs) // len(n2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                n1.dig,
                l2mid,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig, lmax, node.dig, style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    async def pause(self):
        self.paused = True
        await click_future(self._continue)
        self.paused = False


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [ ]:
debug_view = w.Output()
gui = GUI()

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial

    _, _, drafted = result.validate()
    assert drafted

    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, pattern, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(pattern)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: ..."
                gui.resolving = resolving
                render_resolution(pattern)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"

            complete, valid, drafted = result.validate()
            render_status(complete, valid, drafted)
            if complete or not valid or not drafted:
                break
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    complete, valid, drafted = result.validate()
    render_status(complete, valid, drafted)
    gui.resolving = f"#{iteration} ENDED"
    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Pattern):
    with hold_canvas():
        gui.targets = res.get("spoilers", set())
        gui.anchors = res.get("anchors", set())
        gui.empties = res.get("space", set())
        if "chain" in res:
            gui.links = set(res["chain"])
        elif "links" in res:
            gui.links = res["links"]
        else:
            gui.links = set()


def render_status(complete: bool, valid: bool, drafted: bool):
    if not valid:
        gui.status = "BROKEN"
    elif complete:
        gui.status = "SOLVED"
    elif not drafted:
        gui.status = "STUCK"
    else:
        gui.status = "..."


def clear_resolution():
    gui.reset_highlights()

In [ ]:
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle
gui.targets = set()
gui.anchors = set()
gui.links = set()
gui.status = "..."

In [ ]:
# links = set(search_hard_2(puzzle)) | set(search_hard_31(puzzle))
links = set(search_hard_31(puzzle))
ing = iter(links)
len(links)

In [ ]:
gui.links = set()
link = next(ing)
gui.links = {link}

In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        naked_singles,
        hidden_singles,
        # onceolver(naked_multiples),
        # onceolver(hidden_multiples),
        # onceolver(locked_triplets),
        onceolver(multichains),
        # randomchoice,
        filtout={"naked_singles", "hidden_singles", "naked_multiples", "hidden_multiples"},
    )
)

In [ ]:
task

In [ ]:
task.cancel()  # it breaks something